# 02 — Co-visitation Recommendation (Polars/PyArrow, checkpoint-resume)

Bản này **không dùng DuckDB**.

Pipeline:

`processed interactions` → stage theo session bucket → tạo co-visitation pairs bằng sliding window → reduce Top-K → tune trên VAL → refit TRAIN+VAL → TEST.

Các bước dài đều lưu checkpoint:
- Stage: lưu theo từng source parquet.
- Pair generation: lưu theo từng session bucket.
- Final reduce: lưu theo từng anchor bucket.
- Candidate generation: lưu theo từng model bucket.
- Tuning: lưu từng trial.

Nếu kernel/máy bị ngắt, chạy lại các cell setup rồi chạy lại cell đang làm dở; phần đã có checkpoint sẽ `SKIP`.

Dữ liệu preprocessing cũ được giữ nguyên. Notebook chỉ thay engine build/evaluate Co-visitation.


In [1]:
# Cell 2 — Imports + paths
from pathlib import Path
import sys
import os
import gc
import json
import math
import time
import shutil
import itertools

import numpy as np
import pandas as pd

try:
    import polars as pl
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError as e:
    raise ImportError(
        "Thiếu Polars/PyArrow. Chạy trong terminal của .venv:\n"
        "pip install -U polars pyarrow"
    ) from e

from IPython.display import display

PROJECT_ROOT = Path(r"D:\MerRec")

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "recommender"
MODEL_DATA_DIR = PROCESSED_ROOT / "model_data"
INTERACTIONS_DIR = MODEL_DATA_DIR / "interactions"

TRAIN_DIR = INTERACTIONS_DIR / "split=train"
VAL_DIR   = INTERACTIONS_DIR / "split=val"
TEST_DIR  = INTERACTIONS_DIR / "split=test"

TRAIN_FILES = sorted(TRAIN_DIR.glob("*.parquet"))
VAL_FILES   = sorted(VAL_DIR.glob("*.parquet"))
TEST_FILES  = sorted(TEST_DIR.glob("*.parquet"))

COVIS_ROOT = PROJECT_ROOT / "training" / "checkpoints" / "covisitation"
ENGINE_ROOT = COVIS_ROOT / "polars_v2"

STAGE_ROOT = ENGINE_ROOT / "stage"
MODEL_ROOT = ENGINE_ROOT / "models"
EVAL_ROOT = ENGINE_ROOT / "eval"
TUNING_ROOT = ENGINE_ROOT / "tuning"
RESULTS_ROOT = ENGINE_ROOT / "results"

for p in [COVIS_ROOT, ENGINE_ROOT, STAGE_ROOT, MODEL_ROOT, EVAL_ROOT, TUNING_ROOT, RESULTS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("Polars :", pl.__version__)
print("PyArrow:", pa.__version__)
print("TRAIN files:", len(TRAIN_FILES))
print("VAL files  :", len(VAL_FILES))
print("TEST files :", len(TEST_FILES))
print("Output     :", ENGINE_ROOT)


Polars : 1.44.2
PyArrow: 25.0.1
TRAIN files: 1
VAL files  : 1
TEST files : 1
Output     : D:\MerRec\training\checkpoints\covisitation\polars_v2


In [2]:
# Cell 3 — Config
# ===== model =====
SESSION_BUCKETS = 128
ANCHOR_BUCKETS = 128

STAGE_BATCH_ROWS = 250_000

COVIS_WINDOW = 5
MAX_SESSION_LENGTH = 50

# Prune cục bộ để pair checkpoint không phình quá lớn.
# Final model vẫn giữ MODEL_TOPK=250.
PARTIAL_TOPK = 1000
MODEL_TOPK = 250

DISTANCE_POWER = 0.75
HASH_SEED = 20260916

# ===== tuning / eval =====
TUNE_USERS = 2_000
FINAL_EVAL_USERS = 5_000

MAX_HISTORY_FOR_TUNING = 30
HISTORY_OPTIONS = [5, 10, 20, 30]
DECAY_OPTIONS = [0.80, 0.90, 0.97]
REVERSE_WEIGHT_OPTIONS = [0.40, 0.70, 1.00]
POPULARITY_PENALTY_OPTIONS = [0.00, 0.10, 0.20, 0.30]

TUNE_K = 20
K_LIST = [10, 20]

# ===== safety / resume =====
MIN_FREE_GB = 15.0

FORCE_RESTAGE_TRAIN = False
FORCE_RESTAGE_VAL = False
FORCE_REBUILD_TRAIN_MODEL = False
FORCE_REBUILD_FINAL_MODEL = False
FORCE_REBUILD_CANDIDATES = False
FORCE_RETUNE = False

# Xóa CHỈ temp DuckDB cũ từ lần crash trước, không đụng processed data/checkpoint.
CLEAN_OLD_DUCKDB_TEMP = True

# Sau khi VAL tuning xong, pair intermediate của TRAIN có thể bỏ để nhường disk
# trước khi build TRAIN+VAL. Model TRAIN đã hoàn chỉnh vẫn được giữ.
CLEAN_TRAIN_PAIR_INTERMEDIATE_BEFORE_REFIT = True

# Để False khi đang phát triển. Khi mọi thứ hoàn tất có thể đổi True rồi chạy Cell cuối.
CLEAN_INTERMEDIATE_AFTER_COMPLETE = False

print("SESSION_BUCKETS:", SESSION_BUCKETS)
print("ANCHOR_BUCKETS :", ANCHOR_BUCKETS)
print("PARTIAL_TOPK   :", PARTIAL_TOPK)
print("MODEL_TOPK     :", MODEL_TOPK)


SESSION_BUCKETS: 128
ANCHOR_BUCKETS : 128
PARTIAL_TOPK   : 1000
MODEL_TOPK     : 250


In [3]:
# Cell 4 — Preflight + dọn temp DuckDB cũ an toàn
def folder_size_gb(path: Path) -> float:
    if not path.exists():
        return 0.0
    total = 0
    for p in path.rglob("*"):
        try:
            if p.is_file():
                total += p.stat().st_size
        except OSError:
            pass
    return total / (1024 ** 3)

def free_gb(path: Path = PROJECT_ROOT) -> float:
    return shutil.disk_usage(path).free / (1024 ** 3)

OLD_DUCKDB_TEMPS = [
    PROCESSED_ROOT / "model_specific" / "duckdb_temp",
    PROCESSED_ROOT / "model_specific" / "covis_duckdb_temp",
]

print("Free disk before cleanup:", f"{free_gb():.2f} GB")

if CLEAN_OLD_DUCKDB_TEMP:
    for p in OLD_DUCKDB_TEMPS:
        if p.exists():
            size = folder_size_gb(p)
            try:
                shutil.rmtree(p)
                print(f"🧹 Removed old DuckDB temp: {p} ({size:.2f} GB)")
            except Exception as e:
                print(f"⚠️ Không xóa được {p}: {e}")
                print("   Nếu đang bị lock, shutdown/restart kernel rồi chạy lại Cell 4.")

print("Free disk after cleanup :", f"{free_gb():.2f} GB")

if not TRAIN_FILES:
    raise FileNotFoundError(f"Không thấy TRAIN parquet tại {TRAIN_DIR}")
if not VAL_FILES:
    raise FileNotFoundError(f"Không thấy VAL parquet tại {VAL_DIR}")
if not TEST_FILES:
    raise FileNotFoundError(f"Không thấy TEST parquet tại {TEST_DIR}")

train_schema = pq.ParquetFile(TRAIN_FILES[0]).schema_arrow
train_cols = set(train_schema.names)

if "sid" in train_cols:
    SESSION_COL = "sid"
elif "session_id" in train_cols:
    SESSION_COL = "session_id"
elif "session" in train_cols:
    SESSION_COL = "session"
else:
    raise RuntimeError(
        "Không tìm thấy session column. "
        f"Columns hiện có: {sorted(train_cols)}"
    )

required_train = {"user_id", "item_id", "ts", "event_weight", SESSION_COL}
missing = sorted(required_train - train_cols)
if missing:
    raise RuntimeError(f"TRAIN thiếu columns: {missing}")

target_schema = pq.ParquetFile(VAL_FILES[0]).schema_arrow
target_cols = set(target_schema.names)
required_targets = {
    "user_id", "item_id", "event_group",
    "is_strong_positive", "is_purchase"
}
missing_targets = sorted(required_targets - target_cols)
if missing_targets:
    raise RuntimeError(f"VAL/TEST thiếu target columns: {missing_targets}")

print("✅ PREFLIGHT OK")
print("Session column:", SESSION_COL)
print("Free disk      :", f"{free_gb():.2f} GB")


Free disk before cleanup: 74.99 GB
Free disk after cleanup : 74.99 GB
✅ PREFLIGHT OK
Session column: session_id
Free disk      : 74.99 GB


In [4]:
# Cell 5 — Common helpers
def require_free_space(min_gb=MIN_FREE_GB):
    f = free_gb()
    if f < min_gb:
        raise RuntimeError(
            f"Ổ D chỉ còn {f:.2f} GB. "
            f"Cần tối thiểu {min_gb:.2f} GB trước khi tiếp tục."
        )

def atomic_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)

def atomic_write_parquet(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp.parquet")
    if tmp.exists():
        tmp.unlink()
    df.write_parquet(tmp, compression="zstd")
    os.replace(tmp, path)

def valid_parquet(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        meta = pq.ParquetFile(path).metadata
        return meta is not None
    except Exception:
        return False

def collect_lazy(lf):
    # Polars mới: engine="streaming"; Polars cũ: streaming=True
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect(streaming=True)

def hash_bucket_expr(cols, n_buckets, alias):
    txt = pl.concat_str(
        [pl.col(c).cast(pl.Utf8) for c in cols],
        separator="|",
        ignore_nulls=False,
    )
    return (
        (txt.hash(seed=HASH_SEED) % int(n_buckets))
        .cast(pl.UInt16)
        .alias(alias)
    )

def list_split_files(split):
    split = split.lower()
    if split == "train":
        return TRAIN_FILES
    if split == "val":
        return VAL_FILES
    if split == "test":
        return TEST_FILES
    raise ValueError(split)

def scan_files(paths):
    paths = [str(Path(p)) for p in paths]
    if not paths:
        raise ValueError("Không có parquet path để scan")
    return pl.scan_parquet(paths)

def scan_splits(splits):
    paths = []
    for s in splits:
        paths.extend(list_split_files(s))
    return scan_files(paths)

def model_bucket_files(model_dir):
    model_dir = Path(model_dir)
    return sorted((model_dir / "model").glob("bucket_*.parquet"))

def parquet_rows(paths):
    total = 0
    for p in paths:
        try:
            total += pq.ParquetFile(p).metadata.num_rows
        except Exception:
            pass
    return total

print("✅ Helpers ready")


✅ Helpers ready


In [5]:
# Cell 6 — Stage interactions theo session bucket, có checkpoint từng source file
def stage_split(split, source_files, force=False):
    split = split.lower()
    split_root = STAGE_ROOT / split
    split_root.mkdir(parents=True, exist_ok=True)

    if force and split_root.exists():
        print(f"⚠️ FORCE restage {split}: xóa stage cũ")
        shutil.rmtree(split_root)
        split_root.mkdir(parents=True, exist_ok=True)

    columns = ["user_id", SESSION_COL, "item_id", "ts", "event_weight"]

    print("=" * 72)
    print(f"STAGE {split.upper()} — {len(source_files)} source parquet")
    print("=" * 72)

    for source_idx, src in enumerate(source_files):
        final_dir = split_root / f"source_{source_idx:05d}"
        success_path = final_dir / "_SUCCESS.json"

        if success_path.exists():
            print(f"✅ {split} source {source_idx+1}/{len(source_files)} SKIP")
            continue

        require_free_space()

        # Nếu lần trước chết giữa source này: chỉ làm lại đúng source này.
        tmp_dir = split_root / f"source_{source_idx:05d}_tmp"
        if tmp_dir.exists():
            shutil.rmtree(tmp_dir, ignore_errors=True)
        if final_dir.exists():
            shutil.rmtree(final_dir, ignore_errors=True)
        tmp_dir.mkdir(parents=True, exist_ok=True)

        print(f"🚀 {split} source {source_idx+1}/{len(source_files)}: {src.name}")
        t0 = time.time()
        rows_written = 0

        pf = pq.ParquetFile(src)

        for batch_idx, batch in enumerate(
            pf.iter_batches(
                batch_size=STAGE_BATCH_ROWS,
                columns=columns,
                use_threads=True,
            )
        ):
            df = pl.from_arrow(batch)

            df = (
                df.select([
                    pl.col("user_id").cast(pl.Utf8),
                    pl.col(SESSION_COL).cast(pl.Utf8).alias("session_id"),
                    pl.col("item_id").cast(pl.Utf8),
                    pl.col("ts"),
                    pl.col("event_weight").cast(pl.Float32).fill_null(1.0),
                ])
                .filter(
                    pl.col("user_id").is_not_null()
                    & pl.col("session_id").is_not_null()
                    & pl.col("item_id").is_not_null()
                    & pl.col("ts").is_not_null()
                )
                .with_columns(
                    hash_bucket_expr(
                        ["user_id", "session_id"],
                        SESSION_BUCKETS,
                        "session_bucket",
                    )
                )
            )

            if df.height == 0:
                continue

            parts = df.partition_by(
                "session_bucket",
                as_dict=True,
                maintain_order=False,
            )

            for key, part in parts.items():
                bucket = key[0] if isinstance(key, tuple) else key
                bucket = int(bucket)

                out_dir = tmp_dir / f"bucket_{bucket:03d}"
                out_dir.mkdir(parents=True, exist_ok=True)

                out_file = out_dir / f"part_{batch_idx:05d}.parquet"

                part.drop("session_bucket").write_parquet(
                    out_file,
                    compression="zstd",
                )

            rows_written += df.height

            if (batch_idx + 1) % 5 == 0:
                print(
                    f"   batch {batch_idx+1:,} | "
                    f"rows {rows_written:,} | "
                    f"free {free_gb():.1f} GB"
                )

            del df, parts, batch
            gc.collect()

        # Chỉ đánh dấu thành công sau khi source hoàn tất.
        tmp_dir.rename(final_dir)
        atomic_json(
            {
                "split": split,
                "source_index": source_idx,
                "source": str(src),
                "rows": rows_written,
                "completed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            },
            success_path,
        )

        print(
            f"💾 SAVED {split} source {source_idx+1}/{len(source_files)} "
            f"| rows={rows_written:,} | {(time.time()-t0)/60:.2f} min"
        )

    atomic_json(
        {
            "split": split,
            "sources": len(source_files),
            "session_buckets": SESSION_BUCKETS,
            "complete": True,
        },
        split_root / "_STAGE_COMPLETE.json",
    )

    print(f"✅ STAGE {split.upper()} COMPLETE")
    return split_root


In [6]:
# Cell 7 — Stage TRAIN
TRAIN_STAGE_ROOT = stage_split(
    "train",
    TRAIN_FILES,
    force=FORCE_RESTAGE_TRAIN,
)


STAGE TRAIN — 1 source parquet
✅ train source 1/1 SKIP
✅ STAGE TRAIN COMPLETE


In [7]:
# Cell 8 — Build Co-visitation model
# Windows-safe + checkpoint/resume
# KHÔNG rename folder

# ============================================================
# HELPERS
# ============================================================

def safe_rmtree(path, retries=8, delay=0.5):
    """
    Xóa folder an toàn hơn trên Windows.
    Dùng khi bucket chưa hoàn thành hoặc force rebuild.
    """
    path = Path(path)

    if not path.exists():
        return

    last_error = None

    for attempt in range(retries):
        try:
            shutil.rmtree(path)
            return

        except PermissionError as e:
            last_error = e

            gc.collect()

            time.sleep(
                delay * (attempt + 1)
            )

        except FileNotFoundError:
            return

    raise RuntimeError(
        f"Không thể xóa folder sau {retries} lần:\n"
        f"{path}\n"
        f"Lỗi cuối: {last_error}"
    )


def stage_bucket_paths(
    stage_splits,
    bucket,
):
    paths = []

    for split in stage_splits:

        split_root = (
            STAGE_ROOT
            / split
        )

        paths.extend(
            sorted(
                split_root.glob(
                    f"source_*/"
                    f"bucket_{bucket:03d}/"
                    f"*.parquet"
                )
            )
        )

    return paths


# ============================================================
# PAIR GENERATION
# ============================================================

def generate_pairs_for_bucket(events):

    if events.height == 0:
        return None


    # ========================================================
    # DEDUPE
    # cùng user + session + item
    # ========================================================

    events = (
        events.lazy()

        .group_by(
            [
                "user_id",
                "session_id",
                "item_id",
            ]
        )

        .agg([
            pl.col("ts")
            .min()
            .alias("first_ts"),

            pl.col("event_weight")
            .max()
            .alias("event_weight"),
        ])

        .collect()
    )


    if events.height == 0:
        return None


    # ========================================================
    # SORT SESSION
    # ========================================================

    events = events.sort(
        [
            "user_id",
            "session_id",
            "first_ts",
            "item_id",
        ]
    )


    # ========================================================
    # LIMIT SESSION LENGTH
    # tránh session cực dài
    # ========================================================

    events = (
        events

        .group_by(
            [
                "user_id",
                "session_id",
            ],
            maintain_order=True,
        )

        .head(
            MAX_SESSION_LENGTH
        )
    )


    pair_chunks = []

    group_keys = [
        "user_id",
        "session_id",
    ]


    # ========================================================
    # SLIDING WINDOW
    # ========================================================

    for distance in range(
        1,
        COVIS_WINDOW + 1,
    ):

        shifted = (
            events

            .with_columns([

                pl.col("item_id")
                .shift(-distance)
                .over(group_keys)
                .alias("dst_item_id"),


                pl.col("event_weight")
                .shift(-distance)
                .over(group_keys)
                .alias("dst_event_weight"),

            ])

            .filter(

                pl.col(
                    "dst_item_id"
                ).is_not_null()

                &

                (
                    pl.col("item_id")
                    !=
                    pl.col("dst_item_id")
                )
            )
        )


        if shifted.height == 0:
            continue


        denom = (
            float(distance)
            **
            float(DISTANCE_POWER)
        )


        shifted = shifted.with_columns(

            (
                pl.col("event_weight")
                .cast(pl.Float64)

                *

                pl.col("dst_event_weight")
                .cast(pl.Float64)

                /

                denom
            )

            .alias(
                "_pair_score"
            )
        )


        # ====================================================
        # FORWARD
        # ====================================================

        fwd = shifted.select([

            pl.col("item_id")
            .alias("src_item_id"),

            pl.col("dst_item_id"),

            pl.col("_pair_score")
            .alias("forward_score"),

            pl.lit(
                0.0,
                dtype=pl.Float64,
            )
            .alias("reverse_score"),

            pl.lit(
                1,
                dtype=pl.UInt32,
            )
            .alias("forward_count"),

            pl.lit(
                0,
                dtype=pl.UInt32,
            )
            .alias("reverse_count"),

            pl.col("dst_event_weight")
            .cast(pl.Float64)
            .alias("dst_strength"),
        ])


        # ====================================================
        # REVERSE
        # ====================================================

        rev = shifted.select([

            pl.col("dst_item_id")
            .alias("src_item_id"),

            pl.col("item_id")
            .alias("dst_item_id"),

            pl.lit(
                0.0,
                dtype=pl.Float64,
            )
            .alias("forward_score"),

            pl.col("_pair_score")
            .alias("reverse_score"),

            pl.lit(
                0,
                dtype=pl.UInt32,
            )
            .alias("forward_count"),

            pl.lit(
                1,
                dtype=pl.UInt32,
            )
            .alias("reverse_count"),

            pl.col("event_weight")
            .cast(pl.Float64)
            .alias("dst_strength"),
        ])


        # ====================================================
        # AGGREGATE TRONG DISTANCE HIỆN TẠI
        # ====================================================

        part = (

            pl.concat(
                [fwd, rev],
                how="vertical_relaxed",
            )

            .group_by(
                [
                    "src_item_id",
                    "dst_item_id",
                ]
            )

            .agg([

                pl.col(
                    "forward_score"
                ).sum(),

                pl.col(
                    "reverse_score"
                ).sum(),

                pl.col(
                    "forward_count"
                ).sum(),

                pl.col(
                    "reverse_count"
                ).sum(),

                pl.col(
                    "dst_strength"
                ).sum(),
            ])
        )


        pair_chunks.append(
            part
        )


        del (
            shifted,
            fwd,
            rev,
            part,
        )

        gc.collect()


    # ========================================================
    # NO PAIRS
    # ========================================================

    if not pair_chunks:
        return None


    # ========================================================
    # MERGE ALL WINDOW DISTANCES
    # ========================================================

    pairs = (

        pl.concat(
            pair_chunks,
            how="vertical_relaxed",
        )

        .group_by(
            [
                "src_item_id",
                "dst_item_id",
            ]
        )

        .agg([

            pl.col(
                "forward_score"
            ).sum(),

            pl.col(
                "reverse_score"
            ).sum(),

            pl.col(
                "forward_count"
            ).sum(),

            pl.col(
                "reverse_count"
            ).sum(),

            pl.col(
                "dst_strength"
            ).sum(),
        ])

        .with_columns([

            (
                pl.col("forward_count")
                +
                pl.col("reverse_count")
            )
            .alias(
                "_support"
            ),

            (
                pl.col("forward_score")
                +
                pl.col("reverse_score")
            )
            .alias(
                "_partial_score"
            ),

        ])

        .sort(

            [
                "src_item_id",
                "_partial_score",
                "_support",
                "dst_item_id",
            ],

            descending=[
                False,
                True,
                True,
                False,
            ],
        )

        .group_by(
            "src_item_id",
            maintain_order=True,
        )

        .head(
            PARTIAL_TOPK
        )

        .drop([
            "_support",
            "_partial_score",
        ])

        .with_columns(

            hash_bucket_expr(
                ["src_item_id"],
                ANCHOR_BUCKETS,
                "anchor_bucket",
            )
        )
    )


    del pair_chunks

    gc.collect()


    return pairs


# ============================================================
# BUILD SESSION PAIR BUCKETS
# CHECKPOINT TỪNG BUCKET
# ============================================================

def build_pair_buckets(
    run_name,
    stage_splits,
    force=False,
):

    run_root = (
        MODEL_ROOT
        / run_name
    )


    pair_root = (
        run_root
        / "pairs"
    )


    # ========================================================
    # FORCE
    # ========================================================

    if force and pair_root.exists():

        print(
            f"⚠️ FORCE rebuild pairs: "
            f"{run_name}"
        )

        safe_rmtree(
            pair_root
        )


    pair_root.mkdir(
        parents=True,
        exist_ok=True,
    )


    # ========================================================
    # EACH SESSION BUCKET
    # ========================================================

    for bucket in range(
        SESSION_BUCKETS
    ):

        final_dir = (
            pair_root
            / f"session_{bucket:03d}"
        )


        success_path = (
            final_dir
            / "_SUCCESS.json"
        )


        # ====================================================
        # COMPLETED -> SKIP
        # ====================================================

        if success_path.exists():

            print(
                f"✅ {run_name} pair "
                f"{bucket+1}/"
                f"{SESSION_BUCKETS} "
                f"SKIP"
            )

            continue


        # ====================================================
        # INCOMPLETE BUCKET FROM PREVIOUS CRASH
        #
        # Không có SUCCESS => chỉ bucket này không hợp lệ.
        # Xóa nó rồi build lại.
        # ====================================================

        if final_dir.exists():

            print(
                f"♻️ {run_name} pair "
                f"{bucket+1}/"
                f"{SESSION_BUCKETS}: "
                f"found incomplete checkpoint "
                f"-> rebuild bucket"
            )

            safe_rmtree(
                final_dir
            )


        final_dir.mkdir(
            parents=True,
            exist_ok=True,
        )


        require_free_space()


        # ====================================================
        # INPUT
        # ====================================================

        input_paths = stage_bucket_paths(
            stage_splits,
            bucket,
        )


        print()

        print(
            f"🚀 {run_name} pair "
            f"{bucket+1}/"
            f"{SESSION_BUCKETS} "
            f"| input files="
            f"{len(input_paths)}"
        )


        t0 = time.time()


        # ====================================================
        # EMPTY BUCKET
        # ====================================================

        if not input_paths:

            atomic_json(
                {
                    "bucket":
                        bucket,

                    "rows":
                        0,

                    "input_files":
                        0,

                    "empty":
                        True,

                    "completed_at":
                        time.strftime(
                            "%Y-%m-%d %H:%M:%S"
                        ),
                },

                success_path,
            )


            print(
                f"💾 {run_name} pair "
                f"{bucket+1}/"
                f"{SESSION_BUCKETS} "
                f"| EMPTY"
            )


            continue


        # ====================================================
        # LOAD STAGED EVENTS
        # ====================================================

        events = collect_lazy(

            scan_files(
                input_paths
            )

            .select([
                "user_id",
                "session_id",
                "item_id",
                "ts",
                "event_weight",
            ])
        )


        # ====================================================
        # GENERATE COVIS PAIRS
        # ====================================================

        pairs = (
            generate_pairs_for_bucket(
                events
            )
        )


        rows_out = 0


        # ====================================================
        # SAVE BY ANCHOR BUCKET
        # ====================================================

        if (
            pairs is not None
            and
            pairs.height > 0
        ):

            rows_out = (
                pairs.height
            )


            parts = (
                pairs.partition_by(
                    "anchor_bucket",
                    as_dict=True,
                    maintain_order=False,
                )
            )


            for key, part in (
                parts.items()
            ):

                anchor = (
                    key[0]
                    if isinstance(
                        key,
                        tuple,
                    )
                    else key
                )


                anchor = int(
                    anchor
                )


                out_file = (
                    final_dir
                    /
                    f"anchor_{anchor:03d}.parquet"
                )


                # atomic_write_parquet:
                # file chỉ xuất hiện hoàn chỉnh
                atomic_write_parquet(

                    part.drop(
                        "anchor_bucket"
                    ),

                    out_file,
                )


            del (
                parts,
                pairs,
            )


        # ====================================================
        # RELEASE MEMORY / WINDOWS FILE HANDLES
        # ====================================================

        del events

        gc.collect()

        time.sleep(
            0.1
        )


        # ====================================================
        # SUCCESS MARKER
        #
        # RẤT QUAN TRỌNG:
        # chỉ được tạo SAU KHI toàn bộ parquet đã ghi xong.
        # ====================================================

        atomic_json(
            {
                "bucket":
                    bucket,

                "rows":
                    rows_out,

                "input_files":
                    len(
                        input_paths
                    ),

                "completed_at":
                    time.strftime(
                        "%Y-%m-%d %H:%M:%S"
                    ),
            },

            success_path,
        )


        print(
            f"💾 {run_name} pair "
            f"{bucket+1}/"
            f"{SESSION_BUCKETS} "
            f"| rows={rows_out:,} "
            f"| {(time.time()-t0)/60:.2f} min "
            f"| free={free_gb():.1f} GB"
        )


        gc.collect()


    # ========================================================
    # ALL PAIR BUCKETS COMPLETE
    # ========================================================

    completed = sum(

        1

        for bucket in range(
            SESSION_BUCKETS
        )

        if (
            pair_root
            / f"session_{bucket:03d}"
            / "_SUCCESS.json"
        ).exists()
    )


    if completed != SESSION_BUCKETS:

        raise RuntimeError(
            f"{run_name}: pair checkpoint "
            f"chỉ hoàn thành "
            f"{completed}/"
            f"{SESSION_BUCKETS}"
        )


    atomic_json(
        {
            "run_name":
                run_name,

            "stage_splits":
                stage_splits,

            "session_buckets":
                SESSION_BUCKETS,

            "completed_buckets":
                completed,

            "complete":
                True,

            "completed_at":
                time.strftime(
                    "%Y-%m-%d %H:%M:%S"
                ),
        },

        pair_root
        / "_PAIRS_COMPLETE.json",
    )


    return pair_root


# ============================================================
# REDUCE FINAL MODEL BUCKETS
# CHECKPOINT TỪNG ANCHOR
# ============================================================

def reduce_model_buckets(
    run_name,
    force=False,
):

    run_root = (
        MODEL_ROOT
        / run_name
    )


    pair_root = (
        run_root
        / "pairs"
    )


    model_dir = (
        run_root
        / "model"
    )


    # ========================================================
    # FORCE
    # ========================================================

    if force and model_dir.exists():

        print(
            f"⚠️ FORCE rebuild final "
            f"model buckets: {run_name}"
        )


        safe_rmtree(
            model_dir
        )


    model_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    # ========================================================
    # EACH ANCHOR BUCKET
    # ========================================================

    for anchor in range(
        ANCHOR_BUCKETS
    ):

        out_file = (
            model_dir
            / f"bucket_{anchor:03d}.parquet"
        )


        success_path = (
            model_dir
            / f"bucket_{anchor:03d}.success.json"
        )


        # ====================================================
        # VALID CHECKPOINT
        # ====================================================

        if (
            success_path.exists()
            and
            valid_parquet(
                out_file
            )
        ):

            print(
                f"✅ {run_name} final "
                f"{anchor+1}/"
                f"{ANCHOR_BUCKETS} "
                f"SKIP"
            )

            continue


        # ====================================================
        # INCOMPLETE FINAL BUCKET
        # ====================================================

        if success_path.exists():
            success_path.unlink(
                missing_ok=True
            )


        if out_file.exists():
            out_file.unlink(
                missing_ok=True
            )


        require_free_space()


        input_paths = sorted(

            pair_root.glob(
                f"session_*/"
                f"anchor_{anchor:03d}.parquet"
            )
        )


        print()

        print(
            f"🚀 {run_name} final "
            f"{anchor+1}/"
            f"{ANCHOR_BUCKETS} "
            f"| pair files="
            f"{len(input_paths)}"
        )


        t0 = time.time()


        # ====================================================
        # EMPTY
        # ====================================================

        if not input_paths:

            empty = pl.DataFrame(

                schema={
                    "src_item_id":
                        pl.Utf8,

                    "dst_item_id":
                        pl.Utf8,

                    "forward_score":
                        pl.Float64,

                    "reverse_score":
                        pl.Float64,

                    "forward_count":
                        pl.UInt64,

                    "reverse_count":
                        pl.UInt64,

                    "dst_strength":
                        pl.Float64,

                    "rank":
                        pl.UInt32,
                }
            )


            atomic_write_parquet(
                empty,
                out_file,
            )


            atomic_json(
                {
                    "anchor":
                        anchor,

                    "rows":
                        0,

                    "pair_files":
                        0,

                    "empty":
                        True,

                    "completed_at":
                        time.strftime(
                            "%Y-%m-%d %H:%M:%S"
                        ),
                },

                success_path,
            )


            continue


        # ====================================================
        # REDUCE
        # ====================================================

        combined = collect_lazy(

            scan_files(
                input_paths
            )

            .group_by([
                "src_item_id",
                "dst_item_id",
            ])

            .agg([

                pl.col(
                    "forward_score"
                ).sum(),

                pl.col(
                    "reverse_score"
                ).sum(),

                pl.col(
                    "forward_count"
                ).sum(),

                pl.col(
                    "reverse_count"
                ).sum(),

                pl.col(
                    "dst_strength"
                ).sum(),

            ])
        )


        combined = (

            combined

            .with_columns([

                (
                    pl.col(
                        "forward_score"
                    )
                    +
                    pl.col(
                        "reverse_score"
                    )
                )
                .alias(
                    "_score"
                ),


                (
                    pl.col(
                        "forward_count"
                    )
                    +
                    pl.col(
                        "reverse_count"
                    )
                )
                .alias(
                    "_support"
                ),

            ])

            .sort(

                [
                    "src_item_id",
                    "_score",
                    "_support",
                    "dst_item_id",
                ],

                descending=[
                    False,
                    True,
                    True,
                    False,
                ],
            )

            .group_by(
                "src_item_id",
                maintain_order=True,
            )

            .head(
                MODEL_TOPK
            )

            .with_columns(

                pl.col(
                    "dst_item_id"
                )

                .cum_count()

                .over(
                    "src_item_id"
                )

                .cast(
                    pl.UInt32
                )

                .alias(
                    "rank"
                )
            )

            .drop([
                "_score",
                "_support",
            ])
        )


        rows_out = (
            combined.height
        )


        # ====================================================
        # ATOMIC SAVE
        # ====================================================

        atomic_write_parquet(
            combined,
            out_file,
        )


        del combined

        gc.collect()

        time.sleep(
            0.1
        )


        # ====================================================
        # SUCCESS ONLY AFTER VALID PARQUET
        # ====================================================

        if not valid_parquet(
            out_file
        ):

            raise RuntimeError(
                f"Final parquet lỗi: "
                f"{out_file}"
            )


        atomic_json(
            {
                "anchor":
                    anchor,

                "rows":
                    rows_out,

                "pair_files":
                    len(
                        input_paths
                    ),

                "completed_at":
                    time.strftime(
                        "%Y-%m-%d %H:%M:%S"
                    ),
            },

            success_path,
        )


        print(
            f"💾 {run_name} final "
            f"{anchor+1}/"
            f"{ANCHOR_BUCKETS} "
            f"| rows={rows_out:,} "
            f"| {(time.time()-t0)/60:.2f} min "
            f"| free={free_gb():.1f} GB"
        )


        gc.collect()


    # ========================================================
    # VERIFY FINAL BUCKETS
    # ========================================================

    valid_count = 0


    for anchor in range(
        ANCHOR_BUCKETS
    ):

        p = (
            model_dir
            / f"bucket_{anchor:03d}.parquet"
        )

        s = (
            model_dir
            / f"bucket_{anchor:03d}.success.json"
        )


        if (
            s.exists()
            and
            valid_parquet(p)
        ):

            valid_count += 1


    if valid_count != ANCHOR_BUCKETS:

        raise RuntimeError(
            f"{run_name}: mới có "
            f"{valid_count}/"
            f"{ANCHOR_BUCKETS} "
            f"final bucket hợp lệ"
        )


    # ========================================================
    # MODEL COMPLETE
    # ========================================================

    atomic_json(
        {
            "run_name":
                run_name,

            "model_engine":
                "polars_pyarrow",

            "session_buckets":
                SESSION_BUCKETS,

            "anchor_buckets":
                ANCHOR_BUCKETS,

            "covis_window":
                COVIS_WINDOW,

            "max_session_length":
                MAX_SESSION_LENGTH,

            "partial_topk":
                PARTIAL_TOPK,

            "model_topk":
                MODEL_TOPK,

            "distance_power":
                DISTANCE_POWER,

            "valid_final_buckets":
                valid_count,

            "complete":
                True,

            "completed_at":
                time.strftime(
                    "%Y-%m-%d %H:%M:%S"
                ),
        },

        run_root
        / "_MODEL_COMPLETE.json",
    )


    return model_dir


# ============================================================
# BUILD COMPLETE COVIS MODEL
# ============================================================

def build_covis_model(
    run_name,
    stage_splits,
    force=False,
):

    run_root = (
        MODEL_ROOT
        / run_name
    )


    complete_path = (
        run_root
        / "_MODEL_COMPLETE.json"
    )


    model_dir = (
        run_root
        / "model"
    )


    # ========================================================
    # COMPLETE MODEL -> REUSE
    # ========================================================

    if (
        complete_path.exists()
        and
        not force
    ):

        valid_count = 0


        for anchor in range(
            ANCHOR_BUCKETS
        ):

            p = (
                model_dir
                / f"bucket_{anchor:03d}.parquet"
            )


            s = (
                model_dir
                / f"bucket_{anchor:03d}.success.json"
            )


            if (
                s.exists()
                and
                valid_parquet(p)
            ):

                valid_count += 1


        if valid_count == ANCHOR_BUCKETS:

            print(
                f"✅ {run_name} model "
                f"đã hoàn thành -> REUSE"
            )

            return model_dir


    # ========================================================
    # PAIRS
    # ========================================================

    build_pair_buckets(
        run_name=run_name,
        stage_splits=stage_splits,
        force=force,
    )


    # ========================================================
    # REDUCE
    # ========================================================

    return reduce_model_buckets(
        run_name=run_name,
        force=force,
    )


print(
    "✅ Co-visitation build functions ready "
    "(Windows-safe + checkpoint/resume)"
)

✅ Co-visitation build functions ready (Windows-safe + checkpoint/resume)


In [8]:
# Cell 9 — Build TRAIN-only Co-visitation model
TRAIN_MODEL_DIR = build_covis_model(
    run_name="train",
    stage_splits=["train"],
    force=FORCE_REBUILD_TRAIN_MODEL,
)

train_model_files = model_bucket_files(TRAIN_MODEL_DIR)

print()
print("=" * 72)
print("TRAIN MODEL READY")
print("=" * 72)
print("Model dir :", TRAIN_MODEL_DIR)
print("Buckets   :", len(train_model_files))
print("Rows      :", f"{parquet_rows(train_model_files):,}")
print("Free disk :", f"{free_gb():.2f} GB")


✅ train pair 1/128 SKIP
✅ train pair 2/128 SKIP
✅ train pair 3/128 SKIP
✅ train pair 4/128 SKIP
✅ train pair 5/128 SKIP
✅ train pair 6/128 SKIP
✅ train pair 7/128 SKIP
✅ train pair 8/128 SKIP
✅ train pair 9/128 SKIP
✅ train pair 10/128 SKIP
✅ train pair 11/128 SKIP
✅ train pair 12/128 SKIP
✅ train pair 13/128 SKIP
✅ train pair 14/128 SKIP
✅ train pair 15/128 SKIP
✅ train pair 16/128 SKIP
✅ train pair 17/128 SKIP
✅ train pair 18/128 SKIP
✅ train pair 19/128 SKIP
✅ train pair 20/128 SKIP
✅ train pair 21/128 SKIP
✅ train pair 22/128 SKIP
✅ train pair 23/128 SKIP
✅ train pair 24/128 SKIP
✅ train pair 25/128 SKIP
✅ train pair 26/128 SKIP
✅ train pair 27/128 SKIP
✅ train pair 28/128 SKIP
✅ train pair 29/128 SKIP
✅ train pair 30/128 SKIP
✅ train pair 31/128 SKIP
✅ train pair 32/128 SKIP
✅ train pair 33/128 SKIP
✅ train pair 34/128 SKIP
✅ train pair 35/128 SKIP
✅ train pair 36/128 SKIP
✅ train pair 37/128 SKIP
✅ train pair 38/128 SKIP
✅ train pair 39/128 SKIP
✅ train pair 40/128 SKIP
✅ train p

In [9]:
# Cell 10 — Evaluation cache + candidate base, cũng checkpoint/resume
def eval_cache_dir(tag):
    d = EVAL_ROOT / tag
    d.mkdir(parents=True, exist_ok=True)
    return d

def build_eval_cache(
    tag,
    target_split,
    history_splits,
    n_users,
    force=False,
):
    cache = eval_cache_dir(tag)

    users_path = cache / "users.parquet"
    history_path = cache / "history_top.parquet"
    seen_path = cache / "seen.parquet"
    targets_path = cache / "targets.parquet"
    complete_path = cache / "_CACHE_COMPLETE.json"

    if force and cache.exists():
        shutil.rmtree(cache)
        cache.mkdir(parents=True, exist_ok=True)

    if complete_path.exists() and all(
        valid_parquet(p)
        for p in [users_path, history_path, seen_path, targets_path]
    ):
        print(f"✅ Eval cache {tag} đã có -> REUSE")
        return cache

    # ---------- users ----------
    if not valid_parquet(users_path):
        print(f"⏳ [{tag}] build users cohort...")

        hist_users = (
            scan_splits(history_splits)
            .select(pl.col("user_id").cast(pl.Utf8))
            .filter(pl.col("user_id").is_not_null())
            .unique()
        )

        target_users = (
            scan_splits([target_split])
            .select(pl.col("user_id").cast(pl.Utf8))
            .filter(pl.col("user_id").is_not_null())
            .unique()
        )

        users = collect_lazy(
            target_users
            .join(hist_users, on="user_id", how="inner")
            .with_columns(
                pl.col("user_id").hash(seed=HASH_SEED + 17).alias("_h")
            )
            .sort("_h")
            .head(n_users)
            .drop("_h")
        )

        atomic_write_parquet(users, users_path)
        print(f"💾 [{tag}] users={users.height:,}")

    users = pl.read_parquet(users_path)
    user_ids = users["user_id"].to_list()

    # ---------- history ----------
    if not valid_parquet(history_path) or not valid_parquet(seen_path):
        print(f"⏳ [{tag}] build history/seen...")

        h = collect_lazy(
            scan_splits(history_splits)
            .select([
                pl.col("user_id").cast(pl.Utf8),
                pl.col("item_id").cast(pl.Utf8),
                pl.col("event_weight").cast(pl.Float64),
                pl.col("ts"),
            ])
            .filter(
                pl.col("user_id").is_in(user_ids)
                & pl.col("item_id").is_not_null()
            )
            .group_by(["user_id", "item_id"])
            .agg([
                pl.col("event_weight").sum().alias("seed_strength"),
                pl.col("ts").max().alias("last_ts"),
            ])
        )

        seen = h.select(["user_id", "item_id"]).unique()

        history = (
            h.sort(
                ["user_id", "last_ts", "seed_strength", "item_id"],
                descending=[False, True, True, False],
            )
            .group_by("user_id", maintain_order=True)
            .head(MAX_HISTORY_FOR_TUNING)
            .with_columns(
                pl.col("item_id")
                .cum_count()
                .over("user_id")
                .cast(pl.UInt32)
                .alias("hist_rank")
            )
        )

        atomic_write_parquet(history, history_path)
        atomic_write_parquet(seen, seen_path)

        print(f"💾 [{tag}] history rows={history.height:,}")
        print(f"💾 [{tag}] seen rows   ={seen.height:,}")

        del h, history, seen
        gc.collect()

    # ---------- targets ----------
    if not valid_parquet(targets_path):
        print(f"⏳ [{tag}] build targets...")

        targets = collect_lazy(
            scan_splits([target_split])
            .select([
                pl.col("user_id").cast(pl.Utf8),
                pl.col("item_id").cast(pl.Utf8),
                pl.col("event_group"),
                pl.col("is_strong_positive"),
                pl.col("is_purchase"),
            ])
            .filter(
                pl.col("user_id").is_in(user_ids)
                & pl.col("item_id").is_not_null()
            )
            .unique()
        )

        atomic_write_parquet(targets, targets_path)
        print(f"💾 [{tag}] targets={targets.height:,}")

    atomic_json(
        {
            "tag": tag,
            "target_split": target_split,
            "history_splits": history_splits,
            "n_users": int(pl.read_parquet(users_path).height),
            "complete": True,
        },
        complete_path,
    )

    return cache

def build_candidate_base(
    model_dir,
    cache_dir,
    tag,
    force=False,
):
    model_dir = Path(model_dir)
    cache_dir = Path(cache_dir)

    candidate_dir = cache_dir / "candidate_parts"
    complete_path = candidate_dir / "_COMPLETE.json"

    if force and candidate_dir.exists():
        shutil.rmtree(candidate_dir)

    candidate_dir.mkdir(parents=True, exist_ok=True)

    if complete_path.exists():
        print(f"✅ Candidate base {tag} đã có -> REUSE")
        return candidate_dir

    history = pl.read_parquet(cache_dir / "history_top.parquet")
    seen = pl.read_parquet(cache_dir / "seen.parquet").rename(
        {"item_id": "dst_item_id"}
    )

    history = history.with_columns(
        hash_bucket_expr(
            ["item_id"],
            ANCHOR_BUCKETS,
            "src_bucket",
        )
    )

    print("=" * 72)
    print(f"BUILD CANDIDATE BASE — {tag}")
    print("=" * 72)

    for bucket in range(ANCHOR_BUCKETS):
        out_file = candidate_dir / f"part_{bucket:03d}.parquet"
        success_path = candidate_dir / f"part_{bucket:03d}.success.json"

        if success_path.exists() and (not out_file.exists() or valid_parquet(out_file)):
            print(f"✅ candidate {bucket+1}/{ANCHOR_BUCKETS} SKIP")
            continue

        seeds = history.filter(pl.col("src_bucket") == bucket)

        model_file = model_dir / f"bucket_{bucket:03d}.parquet"

        if seeds.height == 0 or not valid_parquet(model_file):
            if out_file.exists():
                out_file.unlink()
            atomic_json({"bucket": bucket, "rows": 0}, success_path)
            continue

        model = pl.read_parquet(model_file)

        if model.height == 0:
            if out_file.exists():
                out_file.unlink()
            atomic_json({"bucket": bucket, "rows": 0}, success_path)
            continue

        candidates = (
            seeds.join(
                model,
                left_on="item_id",
                right_on="src_item_id",
                how="inner",
            )
            .select([
                "user_id",
                pl.col("dst_item_id"),
                "hist_rank",
                "seed_strength",
                "forward_score",
                "reverse_score",
                "forward_count",
                "reverse_count",
                "dst_strength",
            ])
            .join(
                seen,
                on=["user_id", "dst_item_id"],
                how="anti",
            )
        )

        if candidates.height > 0:
            atomic_write_parquet(candidates, out_file)
            rows = candidates.height
        else:
            if out_file.exists():
                out_file.unlink()
            rows = 0

        atomic_json(
            {"bucket": bucket, "rows": rows},
            success_path,
        )

        print(
            f"💾 candidate {bucket+1}/{ANCHOR_BUCKETS} "
            f"| rows={rows:,} | free={free_gb():.1f} GB"
        )

        del seeds, model, candidates
        gc.collect()

    atomic_json(
        {
            "tag": tag,
            "anchor_buckets": ANCHOR_BUCKETS,
            "complete": True,
        },
        complete_path,
    )

    return candidate_dir

def candidate_part_files(candidate_dir):
    return sorted(Path(candidate_dir).glob("part_*.parquet"))

def build_warm_targets(cache_dir, candidate_dir, force=False):
    cache_dir = Path(cache_dir)
    out = cache_dir / "targets_in_candidate_base.parquet"

    if valid_parquet(out) and not force:
        return pl.read_parquet(out)

    targets = pl.read_parquet(cache_dir / "targets.parquet")

    candidate_files = candidate_part_files(candidate_dir)

    if not candidate_files:
        warm = targets.head(0)
        atomic_write_parquet(warm, out)
        return warm

    target_pairs = targets.select(["user_id", "item_id"]).unique()

    warm_pairs = collect_lazy(
        scan_files(candidate_files)
        .select([
            "user_id",
            pl.col("dst_item_id").alias("item_id"),
        ])
        .join(
            target_pairs.lazy(),
            on=["user_id", "item_id"],
            how="inner",
        )
        .unique()
    )

    warm = targets.join(
        warm_pairs,
        on=["user_id", "item_id"],
        how="inner",
    )

    atomic_write_parquet(warm, out)
    return warm

print("✅ Evaluation/candidate functions ready")


✅ Evaluation/candidate functions ready


In [10]:
# Cell 11 — Metrics + recommend
def target_subset(df, mode):
    mode = mode.upper()

    if isinstance(df, pl.DataFrame):
        x = df
        if mode == "ALL":
            pass
        elif mode == "POSITIVE":
            x = x.filter(
                pl.col("event_group").is_in(
                    ["like", "cart", "offer", "buy_start", "buy_comp"]
                )
            )
        elif mode == "STRONG":
            x = x.filter(pl.col("is_strong_positive") == 1)
        elif mode == "PURCHASE":
            x = x.filter(pl.col("is_purchase") == 1)
        else:
            raise ValueError(mode)
        return x.select(["user_id", "item_id"]).unique().to_pandas()

    x = df
    if mode == "ALL":
        pass
    elif mode == "POSITIVE":
        x = x[x["event_group"].isin(["like", "cart", "offer", "buy_start", "buy_comp"])]
    elif mode == "STRONG":
        x = x[x["is_strong_positive"] == 1]
    elif mode == "PURCHASE":
        x = x[x["is_purchase"] == 1]
    else:
        raise ValueError(mode)
    return x[["user_id", "item_id"]].drop_duplicates()

def target_coverage(all_targets, warm_targets, mode):
    a = target_subset(all_targets, mode)
    w = target_subset(warm_targets, mode)
    return len(w) / len(a) if len(a) else np.nan

def calculate_metrics(predictions, targets, k):
    if len(targets) == 0:
        return {
            f"Recall@{k}": np.nan,
            f"Precision@{k}": np.nan,
            f"HitRate@{k}": np.nan,
            f"NDCG@{k}": np.nan,
            "users_eval": 0,
            "targets": 0,
        }

    gt = targets.groupby("user_id")["item_id"].agg(set).to_dict()

    if len(predictions):
        pred = (
            predictions[predictions["rank"] <= k]
            .sort_values(["user_id", "rank"])
            .groupby("user_id")["item_id"]
            .agg(list)
            .to_dict()
        )
    else:
        pred = {}

    recalls, precisions, hits, ndcgs = [], [], [], []

    for user_id, gt_items in gt.items():
        p = pred.get(user_id, [])[:k]
        flags = [1 if item in gt_items else 0 for item in p]
        n_hits = sum(flags)

        recalls.append(n_hits / len(gt_items))
        precisions.append(n_hits / k)
        hits.append(1.0 if n_hits > 0 else 0.0)

        dcg = sum(
            rel / math.log2(rank + 2)
            for rank, rel in enumerate(flags)
        )
        ideal_hits = min(len(gt_items), k)
        idcg = sum(
            1.0 / math.log2(rank + 2)
            for rank in range(ideal_hits)
        )
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    return {
        f"Recall@{k}": float(np.mean(recalls)),
        f"Precision@{k}": float(np.mean(precisions)),
        f"HitRate@{k}": float(np.mean(hits)),
        f"NDCG@{k}": float(np.mean(ndcgs)),
        "users_eval": len(gt),
        "targets": len(targets),
    }

def recommend_from_candidate_base(
    candidate_dir,
    history_items,
    decay,
    reverse_weight,
    popularity_penalty,
    topk=20,
):
    files = candidate_part_files(candidate_dir)

    if not files:
        return pd.DataFrame(
            columns=["user_id", "item_id", "score", "rank"]
        )

    edge = (
        pl.col("forward_score")
        + float(reverse_weight) * pl.col("reverse_score")
    )

    edge_nonneg = (
        pl.when(edge > 0.0)
        .then(edge)
        .otherwise(0.0)
    )

    seed_nonneg = (
        pl.when(pl.col("seed_strength") > 0.0)
        .then(pl.col("seed_strength"))
        .otherwise(0.0)
    )

    # decay^(hist_rank-1) = exp((hist_rank-1)*ln(decay))
    recency = (
        (
            (pl.col("hist_rank").cast(pl.Float64) - 1.0)
            * math.log(float(decay))
        )
        .exp()
    )

    if float(popularity_penalty) == 0.0:
        pop_denom = pl.lit(1.0)
    else:
        dst_nonneg = (
            pl.when(pl.col("dst_strength") > 0.0)
            .then(pl.col("dst_strength"))
            .otherwise(0.0)
        )
        pop_denom = (1.0 + dst_nonneg).pow(float(popularity_penalty))

    lf = (
        scan_files(files)
        .filter(pl.col("hist_rank") <= int(history_items))
        .with_columns(
            (
                (1.0 + seed_nonneg).log()
                * recency
                * ((1.0 + edge_nonneg).log() / pop_denom)
            ).alias("contribution")
        )
        .group_by(["user_id", "dst_item_id"])
        .agg(pl.col("contribution").sum().alias("score"))
    )

    scored = collect_lazy(lf)

    if scored.height == 0:
        return pd.DataFrame(
            columns=["user_id", "item_id", "score", "rank"]
        )

    ranked = (
        scored
        .sort(
            ["user_id", "score", "dst_item_id"],
            descending=[False, True, False],
        )
        .group_by("user_id", maintain_order=True)
        .head(int(topk))
        .with_columns(
            pl.col("dst_item_id")
            .cum_count()
            .over("user_id")
            .cast(pl.UInt32)
            .alias("rank")
        )
        .rename({"dst_item_id": "item_id"})
        .select(["user_id", "item_id", "score", "rank"])
    )

    return ranked.to_pandas()

print("✅ Metrics/recommend ready")


✅ Metrics/recommend ready


In [11]:
# Cell 12 — VAL cache + candidate base
VAL_CACHE_DIR = build_eval_cache(
    tag="val_tune",
    target_split="val",
    history_splits=["train"],
    n_users=TUNE_USERS,
    force=FORCE_REBUILD_CANDIDATES,
)

VAL_CANDIDATE_DIR = build_candidate_base(
    model_dir=TRAIN_MODEL_DIR,
    cache_dir=VAL_CACHE_DIR,
    tag="VAL",
    force=FORCE_REBUILD_CANDIDATES,
)

val_targets_all_pl = pl.read_parquet(VAL_CACHE_DIR / "targets.parquet")
val_targets_warm_pl = build_warm_targets(
    VAL_CACHE_DIR,
    VAL_CANDIDATE_DIR,
    force=FORCE_REBUILD_CANDIDATES,
)

val_targets_all = val_targets_all_pl.to_pandas()
val_targets_warm = val_targets_warm_pl.to_pandas()

print()
print("VAL candidate coverage:")
for mode in ["ALL", "POSITIVE", "STRONG", "PURCHASE"]:
    cov = target_coverage(val_targets_all, val_targets_warm, mode)
    if pd.notna(cov):
        print(f"  {mode:8s}: {cov:.2%}")
    else:
        print(f"  {mode:8s}: N/A")


⏳ [val_tune] build users cohort...
💾 [val_tune] users=2,000
⏳ [val_tune] build history/seen...
💾 [val_tune] history rows=43,850
💾 [val_tune] seen rows   =181,356
⏳ [val_tune] build targets...
💾 [val_tune] targets=30,868
BUILD CANDIDATE BASE — VAL
💾 candidate 1/128 | rows=31,097 | free=69.8 GB
💾 candidate 2/128 | rows=33,820 | free=69.8 GB
💾 candidate 3/128 | rows=28,437 | free=69.8 GB
💾 candidate 4/128 | rows=25,245 | free=69.8 GB
💾 candidate 5/128 | rows=25,923 | free=69.8 GB
💾 candidate 6/128 | rows=29,431 | free=69.8 GB
💾 candidate 7/128 | rows=27,011 | free=69.8 GB
💾 candidate 8/128 | rows=30,307 | free=69.8 GB
💾 candidate 9/128 | rows=21,363 | free=69.8 GB
💾 candidate 10/128 | rows=27,596 | free=69.8 GB
💾 candidate 11/128 | rows=25,676 | free=69.8 GB
💾 candidate 12/128 | rows=26,574 | free=69.8 GB
💾 candidate 13/128 | rows=27,011 | free=69.8 GB
💾 candidate 14/128 | rows=25,771 | free=69.8 GB
💾 candidate 15/128 | rows=28,427 | free=69.8 GB
💾 candidate 16/128 | rows=27,151 | free=69

In [12]:
# Cell 13 — Tune VAL; mỗi trial lưu riêng nên interrupt vẫn resume
BEST_CONFIG_PATH = TUNING_ROOT / "best_config.json"
LEADERBOARD_PATH = TUNING_ROOT / "tuning_leaderboard.csv"
TRIAL_DIR = TUNING_ROOT / "trials"
TRIAL_DIR.mkdir(parents=True, exist_ok=True)

def trial_key(cfg):
    return (
        f"S{int(cfg.get('stage', 0))}"
        f"_H{int(cfg['history_items']):02d}"
        f"_D{float(cfg['decay']):.4f}"
        f"_R{float(cfg['reverse_weight']):.4f}"
        f"_P{float(cfg['popularity_penalty']):.4f}"
    )

def run_trial(cfg, positive_targets):
    key = trial_key(cfg)
    path = TRIAL_DIR / f"{key}.json"

    if path.exists() and not FORCE_RETUNE:
        return json.loads(path.read_text(encoding="utf-8"))

    print("🚀 trial", key)

    recs = recommend_from_candidate_base(
        VAL_CANDIDATE_DIR,
        history_items=cfg["history_items"],
        decay=cfg["decay"],
        reverse_weight=cfg["reverse_weight"],
        popularity_penalty=cfg["popularity_penalty"],
        topk=TUNE_K,
    )

    m = calculate_metrics(recs, positive_targets, TUNE_K)

    row = {
        **cfg,
        **m,
        "trial_key": key,
    }

    atomic_json(row, path)

    print(
        f"   NDCG@{TUNE_K}={m[f'NDCG@{TUNE_K}']:.6f} | "
        f"Recall@{TUNE_K}={m[f'Recall@{TUNE_K}']:.6f} | "
        f"HR@{TUNE_K}={m[f'HitRate@{TUNE_K}']:.6f}"
    )

    return row

def tune_validation():
    if BEST_CONFIG_PATH.exists() and not FORCE_RETUNE:
        print("✅ best_config.json đã có -> REUSE")
        return json.loads(BEST_CONFIG_PATH.read_text(encoding="utf-8"))

    positive_targets = target_subset(val_targets_all, "POSITIVE")

    stage1_rows = []

    print("=" * 72)
    print("TUNING STAGE 1 — HISTORY + DECAY")
    print("=" * 72)

    for history_items, decay in itertools.product(
        HISTORY_OPTIONS,
        DECAY_OPTIONS,
    ):
        cfg = {
            "history_items": int(history_items),
            "decay": float(decay),
            "reverse_weight": 0.70,
            "popularity_penalty": 0.10,
            "stage": 1,
        }
        stage1_rows.append(run_trial(cfg, positive_targets))

    stage1 = pd.DataFrame(stage1_rows).sort_values(
        [f"NDCG@{TUNE_K}", f"Recall@{TUNE_K}", f"HitRate@{TUNE_K}"],
        ascending=False,
    )

    best_history = int(stage1.iloc[0]["history_items"])
    best_decay = float(stage1.iloc[0]["decay"])

    print("✅ Stage 1 best:", {
        "history_items": best_history,
        "decay": best_decay,
    })

    print()
    print("=" * 72)
    print("TUNING STAGE 2 — DIRECTION + POPULARITY PENALTY")
    print("=" * 72)

    stage2_rows = []

    for reverse_weight, pop_penalty in itertools.product(
        REVERSE_WEIGHT_OPTIONS,
        POPULARITY_PENALTY_OPTIONS,
    ):
        cfg = {
            "history_items": best_history,
            "decay": best_decay,
            "reverse_weight": float(reverse_weight),
            "popularity_penalty": float(pop_penalty),
            "stage": 2,
        }
        stage2_rows.append(run_trial(cfg, positive_targets))

    stage2 = pd.DataFrame(stage2_rows).sort_values(
        [f"NDCG@{TUNE_K}", f"Recall@{TUNE_K}", f"HitRate@{TUNE_K}"],
        ascending=False,
    )

    b = stage2.iloc[0]

    best = {
        "history_items": int(b["history_items"]),
        "decay": float(b["decay"]),
        "reverse_weight": float(b["reverse_weight"]),
        "popularity_penalty": float(b["popularity_penalty"]),
        "selection_metric": f"NDCG@{TUNE_K}_POSITIVE",
        "selection_value": float(b[f"NDCG@{TUNE_K}"]),
        "covis_window": COVIS_WINDOW,
        "max_session_length": MAX_SESSION_LENGTH,
        "partial_topk": PARTIAL_TOPK,
        "model_topk": MODEL_TOPK,
        "distance_power": DISTANCE_POWER,
    }

    leaderboard = pd.concat([stage1, stage2], ignore_index=True)
    leaderboard = leaderboard.sort_values(
        [f"NDCG@{TUNE_K}", f"Recall@{TUNE_K}", f"HitRate@{TUNE_K}"],
        ascending=False,
    )

    leaderboard.to_csv(LEADERBOARD_PATH, index=False)
    atomic_json(best, BEST_CONFIG_PATH)

    print()
    print("=" * 72)
    print("🏆 BEST VALIDATION CONFIG")
    print("=" * 72)
    print(json.dumps(best, indent=2))
    display(leaderboard.head(15))

    return best

best_config = tune_validation()


TUNING STAGE 1 — HISTORY + DECAY
🚀 trial S1_H05_D0.8000_R0.7000_P0.1000
   NDCG@20=0.004501 | Recall@20=0.007196 | HR@20=0.018038
🚀 trial S1_H05_D0.9000_R0.7000_P0.1000
   NDCG@20=0.004229 | Recall@20=0.007035 | HR@20=0.019166
🚀 trial S1_H05_D0.9700_R0.7000_P0.1000
   NDCG@20=0.004443 | Recall@20=0.007430 | HR@20=0.020293
🚀 trial S1_H10_D0.8000_R0.7000_P0.1000
   NDCG@20=0.004558 | Recall@20=0.006659 | HR@20=0.019166
🚀 trial S1_H10_D0.9000_R0.7000_P0.1000
   NDCG@20=0.004320 | Recall@20=0.007178 | HR@20=0.021421
🚀 trial S1_H10_D0.9700_R0.7000_P0.1000
   NDCG@20=0.003677 | Recall@20=0.004875 | HR@20=0.018038
🚀 trial S1_H20_D0.8000_R0.7000_P0.1000
   NDCG@20=0.004836 | Recall@20=0.006820 | HR@20=0.020293
🚀 trial S1_H20_D0.9000_R0.7000_P0.1000
   NDCG@20=0.004798 | Recall@20=0.007219 | HR@20=0.022548
🚀 trial S1_H20_D0.9700_R0.7000_P0.1000
   NDCG@20=0.004522 | Recall@20=0.005496 | HR@20=0.020293
🚀 trial S1_H30_D0.8000_R0.7000_P0.1000
   NDCG@20=0.004836 | Recall@20=0.006820 | HR@20=0.0202

,history_items,decay,reverse_weight,popularity_penalty,stage,Recall@20,Precision@20,HitRate@20,NDCG@20,users_eval,targets,trial_key
12,30,0.90,0.4,0.1,2,0.006984,0.001240,0.022548,0.004987,887,4212,S2_H30_D0.9000_R0.4000_P0.1000
13,30,0.90,0.4,0.0,2,0.007360,0.001297,0.023675,0.004986,887,4212,S2_H30_D0.9000_R0.4000_P0.0000
0,30,0.90,0.7,0.1,1,0.007219,0.001184,0.022548,0.004943,887,4212,S1_H30_D0.9000_R0.7000_P0.1000
14,30,0.90,0.7,0.1,2,0.007219,0.001184,0.022548,0.004943,887,4212,S2_H30_D0.9000_R0.7000_P0.1000
15,30,0.90,1.0,0.0,2,0.007332,0.001240,0.022548,0.004943,887,4212,S2_H30_D0.9000_R1.0000_P0.0000
16,30,0.90,0.7,0.0,2,0.007332,0.001240,0.022548,0.004938,887,4212,S2_H30_D0.9000_R0.7000_P0.0000
17,30,0.90,0.4,0.2,2,0.006718,0.001127,0.022548,0.004933,887,4212,S2_H30_D0.9000_R0.4000_P0.2000
1,30,0.97,0.7,0.1,1,0.006749,0.001466,0.021421,0.004905,887,4212,S1_H30_D0.9700_R0.7000_P0.1000
2,20,0.80,0.7,0.1,1,0.006820,0.001071,0.020293,0.004836,887,4212,S1_H20_D0.8000_R0.7000_P0.1000
3,30,0.80,0.7,0.1,1,0.006820,0.001071,0.020293,0.004836,887,4212,S1_H30_D0.8000_R0.7000_P0.1000


In [13]:
# Cell 14 — VAL metrics
VAL_RESULTS_PATH = RESULTS_ROOT / "validation_metrics.csv"

val_predictions = recommend_from_candidate_base(
    VAL_CANDIDATE_DIR,
    history_items=best_config["history_items"],
    decay=best_config["decay"],
    reverse_weight=best_config["reverse_weight"],
    popularity_penalty=best_config["popularity_penalty"],
    topk=max(K_LIST),
)

val_rows = []

for mode in ["ALL", "POSITIVE", "STRONG", "PURCHASE"]:
    gt = target_subset(val_targets_all, mode)

    row = {
        "split": "VAL",
        "target_mode": mode,
        "candidate_coverage": target_coverage(
            val_targets_all,
            val_targets_warm,
            mode,
        ),
    }

    for k in K_LIST:
        row.update(
            calculate_metrics(
                val_predictions,
                gt,
                k,
            )
        )

    val_rows.append(row)

val_metrics = pd.DataFrame(val_rows)
val_metrics.to_csv(VAL_RESULTS_PATH, index=False)

print("=" * 72)
print("VALIDATION METRICS — BEST CONFIG")
print("=" * 72)
display(val_metrics)
print("Saved:", VAL_RESULTS_PATH)


VALIDATION METRICS — BEST CONFIG


,split,target_mode,candidate_coverage,Recall@10,Precision@10,HitRate@10,NDCG@10,users_eval,targets,Recall@20,Precision@20,HitRate@20,NDCG@20
0,VAL,ALL,0.059448,0.005638,0.004050,0.037500,0.005958,2000,27604,0.008819,0.003475,0.053000,0.006836
1,VAL,POSITIVE,0.060304,0.004134,0.001691,0.016911,0.004136,887,4212,0.006984,0.001240,0.022548,0.004987
2,VAL,STRONG,0.084507,0.015873,0.002041,0.020408,0.014528,147,213,0.015873,0.001020,0.020408,0.014528
3,VAL,PURCHASE,0.000000,0.000000,0.000000,0.000000,0.000000,12,13,0.000000,0.000000,0.000000,0.000000


Saved: D:\MerRec\training\checkpoints\covisitation\polars_v2\results\validation_metrics.csv


In [14]:
# Cell 15 — Stage VAL + refit final TRAIN+VAL model
# TRAIN stage được tái sử dụng; không scan lại TRAIN source.
VAL_STAGE_ROOT = stage_split(
    "val",
    VAL_FILES,
    force=FORCE_RESTAGE_VAL,
)

if CLEAN_TRAIN_PAIR_INTERMEDIATE_BEFORE_REFIT:
    train_pair_dir = MODEL_ROOT / "train" / "pairs"
    train_complete = MODEL_ROOT / "train" / "_MODEL_COMPLETE.json"

    if train_complete.exists() and train_pair_dir.exists():
        size = folder_size_gb(train_pair_dir)
        shutil.rmtree(train_pair_dir, ignore_errors=True)
        print(
            f"🧹 Đã xóa TRAIN pair intermediate sau khi VAL xong: "
            f"{size:.2f} GB"
        )
        print("   TRAIN model bucket vẫn được giữ nguyên.")

FINAL_MODEL_DIR = build_covis_model(
    run_name="train_val",
    stage_splits=["train", "val"],
    force=FORCE_REBUILD_FINAL_MODEL,
)

final_model_files = model_bucket_files(FINAL_MODEL_DIR)

print()
print("=" * 72)
print("FINAL TRAIN+VAL MODEL READY")
print("=" * 72)
print("Model dir :", FINAL_MODEL_DIR)
print("Buckets   :", len(final_model_files))
print("Rows      :", f"{parquet_rows(final_model_files):,}")
print("Free disk :", f"{free_gb():.2f} GB")


STAGE VAL — 1 source parquet
🚀 val source 1/1: data_0.parquet
   batch 5 | rows 1,250,000 | free 69.7 GB
   batch 10 | rows 2,500,000 | free 69.6 GB
   batch 15 | rows 3,750,000 | free 69.6 GB
   batch 20 | rows 5,000,000 | free 69.5 GB
   batch 25 | rows 6,250,000 | free 69.5 GB
   batch 30 | rows 7,500,000 | free 69.4 GB
   batch 35 | rows 8,750,000 | free 69.4 GB
   batch 40 | rows 10,000,000 | free 69.3 GB
   batch 45 | rows 11,250,000 | free 69.2 GB
   batch 50 | rows 12,500,000 | free 69.2 GB
   batch 55 | rows 13,750,000 | free 69.1 GB
   batch 60 | rows 15,000,000 | free 69.1 GB
   batch 65 | rows 16,250,000 | free 69.0 GB
💾 SAVED val source 1/1 | rows=16,605,699 | 0.38 min
✅ STAGE VAL COMPLETE
🧹 Đã xóa TRAIN pair intermediate sau khi VAL xong: 5.61 GB
   TRAIN model bucket vẫn được giữ nguyên.

🚀 train_val pair 1/128 | input files=642
💾 train_val pair 1/128 | rows=5,876,752 | 0.21 min | free=74.6 GB

🚀 train_val pair 2/128 | input files=642
💾 train_val pair 2/128 | rows=5,925,

In [15]:
# Cell 16 — TEST cache + candidate base
TEST_CACHE_DIR = build_eval_cache(
    tag="test_final",
    target_split="test",
    history_splits=["train", "val"],
    n_users=FINAL_EVAL_USERS,
    force=FORCE_REBUILD_CANDIDATES,
)

TEST_CANDIDATE_DIR = build_candidate_base(
    model_dir=FINAL_MODEL_DIR,
    cache_dir=TEST_CACHE_DIR,
    tag="TEST",
    force=FORCE_REBUILD_CANDIDATES,
)

test_targets_all_pl = pl.read_parquet(TEST_CACHE_DIR / "targets.parquet")
test_targets_warm_pl = build_warm_targets(
    TEST_CACHE_DIR,
    TEST_CANDIDATE_DIR,
    force=FORCE_REBUILD_CANDIDATES,
)

test_targets_all = test_targets_all_pl.to_pandas()
test_targets_warm = test_targets_warm_pl.to_pandas()

print()
print("TEST candidate coverage:")
for mode in ["ALL", "POSITIVE", "STRONG", "PURCHASE"]:
    cov = target_coverage(test_targets_all, test_targets_warm, mode)
    if pd.notna(cov):
        print(f"  {mode:8s}: {cov:.2%}")
    else:
        print(f"  {mode:8s}: N/A")


⏳ [test_final] build users cohort...
💾 [test_final] users=5,000
⏳ [test_final] build history/seen...
💾 [test_final] history rows=114,104
💾 [test_final] seen rows   =545,896
⏳ [test_final] build targets...
💾 [test_final] targets=69,520
BUILD CANDIDATE BASE — TEST
💾 candidate 1/128 | rows=80,679 | free=74.6 GB
💾 candidate 2/128 | rows=74,464 | free=74.6 GB
💾 candidate 3/128 | rows=75,282 | free=74.6 GB
💾 candidate 4/128 | rows=68,675 | free=74.6 GB
💾 candidate 5/128 | rows=80,373 | free=74.6 GB
💾 candidate 6/128 | rows=75,091 | free=74.6 GB
💾 candidate 7/128 | rows=73,295 | free=74.6 GB
💾 candidate 8/128 | rows=69,220 | free=74.6 GB
💾 candidate 9/128 | rows=70,508 | free=74.6 GB
💾 candidate 10/128 | rows=69,346 | free=74.6 GB
💾 candidate 11/128 | rows=78,150 | free=74.6 GB
💾 candidate 12/128 | rows=74,048 | free=74.6 GB
💾 candidate 13/128 | rows=72,420 | free=74.6 GB
💾 candidate 14/128 | rows=73,467 | free=74.6 GB
💾 candidate 15/128 | rows=67,767 | free=74.6 GB
💾 candidate 16/128 | rows=

In [16]:
# Cell 17 — FINAL TEST + summary
TEST_RESULTS_PATH = RESULTS_ROOT / "test_metrics.csv"
SUMMARY_PATH = RESULTS_ROOT / "val_test_summary.csv"

test_predictions = recommend_from_candidate_base(
    TEST_CANDIDATE_DIR,
    history_items=best_config["history_items"],
    decay=best_config["decay"],
    reverse_weight=best_config["reverse_weight"],
    popularity_penalty=best_config["popularity_penalty"],
    topk=max(K_LIST),
)

test_rows = []

for mode in ["ALL", "POSITIVE", "STRONG", "PURCHASE"]:
    gt = target_subset(test_targets_all, mode)

    row = {
        "split": "TEST",
        "target_mode": mode,
        "candidate_coverage": target_coverage(
            test_targets_all,
            test_targets_warm,
            mode,
        ),
    }

    for k in K_LIST:
        row.update(
            calculate_metrics(
                test_predictions,
                gt,
                k,
            )
        )

    test_rows.append(row)

test_metrics = pd.DataFrame(test_rows)
test_metrics.to_csv(TEST_RESULTS_PATH, index=False)

summary = pd.concat(
    [val_metrics, test_metrics],
    ignore_index=True,
)
summary.to_csv(SUMMARY_PATH, index=False)

print("=" * 72)
print("🏁 FINAL TEST METRICS")
print("=" * 72)
display(test_metrics)

print()
print("=" * 72)
print("VAL + TEST SUMMARY")
print("=" * 72)
display(summary)

print()
print("BEST CONFIG")
print(json.dumps(best_config, indent=2))

positive = test_metrics[
    test_metrics["target_mode"] == "POSITIVE"
]

if len(positive):
    r = positive.iloc[0]
    print()
    print("POSITIVE TEST")
    print(f"Recall@20  : {r['Recall@20']:.4%}")
    print(f"HitRate@20 : {r['HitRate@20']:.4%}")
    print(f"NDCG@20    : {r['NDCG@20']:.6f}")
    print(f"Coverage   : {r['candidate_coverage']:.2%}")

print()
print("Saved:")
print(" ", TEST_RESULTS_PATH)
print(" ", SUMMARY_PATH)


🏁 FINAL TEST METRICS


,split,target_mode,candidate_coverage,Recall@10,Precision@10,HitRate@10,NDCG@10,users_eval,targets,Recall@20,Precision@20,HitRate@20,NDCG@20
0,TEST,ALL,0.063361,0.007047,0.004800,0.040800,0.007456,5000,61978,0.010523,0.003820,0.058800,0.008200
1,TEST,POSITIVE,0.060919,0.007705,0.001871,0.017800,0.005557,2191,9767,0.012245,0.001575,0.028298,0.006931
2,TEST,STRONG,0.057803,0.008913,0.001070,0.010695,0.004314,374,519,0.009804,0.000668,0.013369,0.004628
3,TEST,PURCHASE,0.000000,0.000000,0.000000,0.000000,0.000000,37,53,0.000000,0.000000,0.000000,0.000000



VAL + TEST SUMMARY


,split,target_mode,candidate_coverage,Recall@10,Precision@10,HitRate@10,NDCG@10,users_eval,targets,Recall@20,Precision@20,HitRate@20,NDCG@20
0,VAL,ALL,0.059448,0.005638,0.004050,0.037500,0.005958,2000,27604,0.008819,0.003475,0.053000,0.006836
1,VAL,POSITIVE,0.060304,0.004134,0.001691,0.016911,0.004136,887,4212,0.006984,0.001240,0.022548,0.004987
2,VAL,STRONG,0.084507,0.015873,0.002041,0.020408,0.014528,147,213,0.015873,0.001020,0.020408,0.014528
3,VAL,PURCHASE,0.000000,0.000000,0.000000,0.000000,0.000000,12,13,0.000000,0.000000,0.000000,0.000000
4,TEST,ALL,0.063361,0.007047,0.004800,0.040800,0.007456,5000,61978,0.010523,0.003820,0.058800,0.008200
5,TEST,POSITIVE,0.060919,0.007705,0.001871,0.017800,0.005557,2191,9767,0.012245,0.001575,0.028298,0.006931
6,TEST,STRONG,0.057803,0.008913,0.001070,0.010695,0.004314,374,519,0.009804,0.000668,0.013369,0.004628
7,TEST,PURCHASE,0.000000,0.000000,0.000000,0.000000,0.000000,37,53,0.000000,0.000000,0.000000,0.000000



BEST CONFIG
{
  "history_items": 30,
  "decay": 0.9,
  "reverse_weight": 0.4,
  "popularity_penalty": 0.1,
  "selection_metric": "NDCG@20_POSITIVE",
  "selection_value": 0.004987018678143363,
  "covis_window": 5,
  "max_session_length": 50,
  "partial_topk": 1000,
  "model_topk": 250,
  "distance_power": 0.75
}

POSITIVE TEST
Recall@20  : 1.2245%
HitRate@20 : 2.8298%
NDCG@20    : 0.006931
Coverage   : 6.09%

Saved:
  D:\MerRec\training\checkpoints\covisitation\polars_v2\results\test_metrics.csv
  D:\MerRec\training\checkpoints\covisitation\polars_v2\results\val_test_summary.csv


In [17]:
# Cell 18 — Optional cleanup sau khi TOÀN BỘ pipeline đã hoàn thành
# Mặc định CLEAN_INTERMEDIATE_AFTER_COMPLETE=False để giữ khả năng resume/rebuild.
if CLEAN_INTERMEDIATE_AFTER_COMPLETE:
    required_complete = [
        MODEL_ROOT / "train_val" / "_MODEL_COMPLETE.json",
        RESULTS_ROOT / "test_metrics.csv",
    ]

    if all(p.exists() for p in required_complete):
        cleanup_targets = [
            STAGE_ROOT,
            MODEL_ROOT / "train_val" / "pairs",
            EVAL_ROOT / "val_tune" / "candidate_parts",
            EVAL_ROOT / "test_final" / "candidate_parts",
        ]

        freed = 0.0

        for p in cleanup_targets:
            if p.exists():
                size = folder_size_gb(p)
                freed += size
                shutil.rmtree(p, ignore_errors=True)
                print(f"🧹 removed {p} ({size:.2f} GB)")

        print(f"✅ Reclaimed khoảng {freed:.2f} GB")
        print("Final model buckets + metrics + best config vẫn được giữ.")
    else:
        print("⚠️ Chưa đủ final artifacts -> KHÔNG cleanup.")
else:
    print("CLEAN_INTERMEDIATE_AFTER_COMPLETE=False -> giữ checkpoint để resume.")


CLEAN_INTERMEDIATE_AFTER_COMPLETE=False -> giữ checkpoint để resume.
